In [7]:
import pandas as pd

In [8]:
data_examples = pd.read_parquet("../data/shopping_queries_dataset_examples.parquet")

In [9]:
data_products = pd.read_parquet("../data/shopping_queries_dataset_products.parquet")

In [10]:
data_sources = pd.read_csv("../data/shopping_queries_dataset_sources.csv")

In [11]:
data_examples.head(1)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train


In [12]:
data_products.head(1)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es


In [13]:
data_examples["product_locale"].nunique()
data_examples["product_locale"].unique()

<ArrowStringArray>
['us', 'es', 'jp']
Length: 3, dtype: str

In [14]:
data_products["product_locale"].nunique()
data_products["product_locale"].unique()

<ArrowStringArray>
['es', 'us', 'jp']
Length: 3, dtype: str

In [15]:
data_examples.tail(4)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
2621284,2621284,�����j�[�h�p�[�x abrasus,130651,B0062EZYIG,jp,E,0,1,test
2621285,2621285,�����j�[�h�p�[�x abrasus,130651,B07H8MWBZN,jp,S,0,1,test
2621286,2621286,�����j�[�h�p�[�x abrasus,130651,B00IZH4T9S,jp,E,0,1,test
2621287,2621287,�����j�[�h�p�[�x abrasus,130651,B07R536DG1,jp,E,0,1,test


In [16]:
data_examples_products = pd.merge(
    data_examples,
    data_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)

In [18]:
data_examples_products.head(1)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,NaN,WhisperCeiling fans feature a totally enclosed...,Panasonic,White


In [19]:
data_examples_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 2621288 entries, 0 to 2621287
Data columns (total 14 columns):
 #   Column                Dtype
---  ------                -----
 0   example_id            int64
 1   query                 str  
 2   query_id              int64
 3   product_id            str  
 4   product_locale        str  
 5   esci_label            str  
 6   small_version         int64
 7   large_version         int64
 8   split                 str  
 9   product_title         str  
 10  product_description   str  
 11  product_bullet_point  str  
 12  product_brand         str  
 13  product_color         str  
dtypes: int64(4), str(10)
memory usage: 3.3 GB


In [20]:
df = data_examples_products[data_examples_products["small_version"] == 1]

In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 1118011 entries, 16 to 2621255
Data columns (total 14 columns):
 #   Column                Non-Null Count    Dtype
---  ------                --------------    -----
 0   example_id            1118011 non-null  int64
 1   query                 1118011 non-null  str  
 2   query_id              1118011 non-null  int64
 3   product_id            1118011 non-null  str  
 4   product_locale        1118011 non-null  str  
 5   esci_label            1118011 non-null  str  
 6   small_version         1118011 non-null  int64
 7   large_version         1118011 non-null  int64
 8   split                 1118011 non-null  str  
 9   product_title         1118011 non-null  str  
 10  product_description   539415 non-null   str  
 11  product_bullet_point  950854 non-null   str  
 12  product_brand         1030301 non-null  str  
 13  product_color         688565 non-null   str  
dtypes: int64(4), str(10)
memory usage: 1.4 GB


In [23]:
df_us = df[df['product_locale'] == 'us'].copy()

In [24]:
df_us.info()

<class 'pandas.DataFrame'>
Index: 601354 entries, 16 to 2618569
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   example_id            601354 non-null  int64
 1   query                 601354 non-null  str  
 2   query_id              601354 non-null  int64
 3   product_id            601354 non-null  str  
 4   product_locale        601354 non-null  str  
 5   esci_label            601354 non-null  str  
 6   small_version         601354 non-null  int64
 7   large_version         601354 non-null  int64
 8   split                 601354 non-null  str  
 9   product_title         601354 non-null  str  
 10  product_description   301110 non-null  str  
 11  product_bullet_point  531226 non-null  str  
 12  product_brand         571704 non-null  str  
 13  product_color         416407 non-null  str  
dtypes: int64(4), str(10)
memory usage: 777.1 MB


In [25]:
df_us['product_locale'].unique()

<ArrowStringArray>
['us']
Length: 1, dtype: str

In [26]:
df_us['small_version'].unique()

array([1])

In [27]:
# Document construction
# Fill NaNs values
cols = ['product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color']
for col in cols:
    df_us[col] = df_us[col].fillna('')

# Concatenation
df_us['product_document'] = (
    df_us['product_title'] + " " +
    df_us['product_description'] + " " +
    df_us['product_bullet_point'] + " " +
    df_us['product_brand'] + " " +
    df_us['product_color']
)

In [28]:
df_us.info()

<class 'pandas.DataFrame'>
Index: 601354 entries, 16 to 2618569
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   example_id            601354 non-null  int64
 1   query                 601354 non-null  str  
 2   query_id              601354 non-null  int64
 3   product_id            601354 non-null  str  
 4   product_locale        601354 non-null  str  
 5   esci_label            601354 non-null  str  
 6   small_version         601354 non-null  int64
 7   large_version         601354 non-null  int64
 8   split                 601354 non-null  str  
 9   product_title         601354 non-null  str  
 10  product_description   601354 non-null  str  
 11  product_bullet_point  601354 non-null  str  
 12  product_brand         601354 non-null  str  
 13  product_color         601354 non-null  str  
 14  product_document      601354 non-null  str  
dtypes: int64(4), str(11)
memory usage: 1.4 GB


In [29]:
import re
import html
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()

STOPWORDS = {
    'i', 'me', 'my', 'a', 'an', 'the', 'and', 'or', 'but', 'are',
    'is', 'was', 'to', 'of', 'in', 'it', 'its', 'this', 'that',
    'for', 'on', 'at', 'be', 'by', 'as', 'up', 'do', 'so', 'if', 's'
}

def clean_text(text):
    # Decode HTML entities and strip tags
    text = html.unescape(str(text))
    text = re.sub(r'<.*?>|&nbsp;', ' ', text)
    # Lowercase
    text = text.lower()
    # Keep alphanumeric only
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Lemmatize + stopword removal
    tokens = [lemmatizer.lemmatize(t) for t in text.split() if t not in STOPWORDS]
    return ' '.join(tokens)

df_us['clean_product_document'] = df_us['product_document'].apply(clean_text)
df_us['clean_query'] = df_us['query'].apply(clean_text)


In [30]:
df_us.head(2)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color,product_document,clean_product_document,clean_query
16,16,!awnmower tires without rims,1,B075SCHMPY,us,I,1,1,train,"RamPro 10"" All Purpose Utility Air Tires/Wheel...","<b>About The Ram-Pro All Purpose Utility 10"" A...",✓ The Ram-Pro Ten Inch ready to install Air Ti...,RamPro,10 Inch,"RamPro 10"" All Purpose Utility Air Tires/Wheel...",rampro 10 all purpose utility air tire wheel w...,awnmower tire without rim
17,17,!awnmower tires without rims,1,B08L3B9B9P,us,E,1,1,train,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,Please check your existing tire Sidewall for t...,MaxAuto,,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,maxauto 2 pack 13x5 00 6 2ply turf mower tract...,awnmower tire without rim


In [31]:
duplicates = df_us[df_us.duplicated(subset=['query_id', 'product_id'], keep=False)]
duplicates.value_counts()

Series([], Name: count, dtype: int64)

In [32]:
df_us['esci_label'].value_counts()

esci_label
E    261527
S    211191
I    101447
C     27189
Name: count, dtype: int64

In [33]:
# Relevance mapping
relevance_map = {
    'e': 1.0,
    's': 0.6,
    'i': 0.1,
    'c': 0.0
}

df_us['relevance_score'] = df_us['esci_label'].str.lower().map(relevance_map)

In [34]:
df_us[['esci_label', 'relevance_score']].head()

,esci_label,relevance_score
16,I,0.1
17,E,1.0
18,I,0.1
19,S,0.6
20,E,1.0


In [35]:
# Create final dataframe with relevant features
df_relevant = df_us[[
    'query_id',
    'product_id',
    'clean_query',
    'clean_product_document',
    'relevance_score',
    'split'
]].copy()

In [36]:
df_relevant.info()

<class 'pandas.DataFrame'>
Index: 601354 entries, 16 to 2618569
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   query_id                601354 non-null  int64  
 1   product_id              601354 non-null  str    
 2   clean_query             601354 non-null  str    
 3   clean_product_document  601354 non-null  str    
 4   relevance_score         601354 non-null  float64
 5   split                   601354 non-null  str    
dtypes: float64(1), int64(1), str(4)
memory usage: 616.1 MB


In [40]:
df_relevant.to_csv("../data/shopping_queries_dataset_final.csv")